# Coleta de dados — Discursos por Parlamentar da Câmara dos Deputados

Este notebook **baixa os discursos de cada deputado** da API pública da Câmara e salva
**um arquivo por deputado** em `discursos/deputado_{id}.csv`. No fim, junta tudo em um único
`discursos_todos.csv`. Esses textos são a entrada da Fase 6 (classificar os discursos por
tema com o BERTimbau e cruzar com as proposições).

- **Fonte:** API de Dados Abertos da Câmara — `https://dadosabertos.camara.leg.br` (pública, sem login).
- **Período:** 2023 a 2026.
- **Um arquivo por deputado** deixa a coleta **retomável**: se parar no meio, ao rodar de
  novo ele **pula** quem já foi baixado.
- Guardamos só os discursos **com texto** (`transcricao` não-vazia) — é o que vamos classificar.
- **Importante:** os dados são coletados ao vivo da fonte oficial. **Não usamos base pronta.**

> Pré-requisito na pasta: `parlamentares.csv` (gerado por `02_coleta_parlamentares.ipynb`).
> Estimativa: ~20–40 min e dezenas de milhares de discursos.

In [ ]:
import requests   # para acessar a API pela internet
import time       # para dar pequenas pausas entre as chamadas
import pandas as pd  # para organizar os dados em tabela
from datetime import date

BASE = "https://dadosabertos.camara.leg.br/api/v2"

def chamar_api(caminho, parametros=None):
    """Faz uma chamada na API e devolve o resultado (em formato JSON).
    Tenta ate 3 vezes caso o servidor falhe."""
    for tentativa in range(3):
        try:
            resposta = requests.get(
                BASE + caminho,
                params=parametros,
                headers={"Accept": "application/json"},
                timeout=30,
            )
            if resposta.status_code == 200:
                return resposta.json()
        except requests.RequestException:
            pass
        time.sleep(1)   # espera 1 segundo e tenta de novo
    return None

print("Pronto! A funcao chamar_api esta carregada.")

Pronto! A funcao chamar_api esta carregada.


In [ ]:
import os

# garante a pasta de saida (nao quebra se ja existir)
os.makedirs("discursos", exist_ok=True)

# colunas fixas: garante que ate deputado SEM discurso gere um arquivo com cabecalho
# (evita o EmptyDataError na hora de juntar)
COLS = ["id_deputado", "nome_deputado", "data", "tipo", "sumario", "transcricao"]

deputados = pd.read_csv("parlamentares.csv", sep=";", encoding="utf-8", dtype=str)
deputados["id"] = deputados["id"].str.replace(r"\.0$", "", regex=True)

ANOS = [2023, 2024, 2025, 2026]

for i, dep in deputados.iterrows():
    caminho = f"discursos/deputado_{dep['id']}.csv"
    if os.path.exists(caminho):          # RETOMADA: pula quem ja foi baixado
        continue

    linhas = []
    for ano in ANOS:
        # 2026 ainda esta em curso: vai ate hoje
        data_fim = date.today().isoformat() if ano == 2026 else f"{ano}-12-31"
        pagina = 1
        while True:
            resposta = chamar_api(f"/deputados/{dep['id']}/discursos", {
                "dataInicio": f"{ano}-01-01",
                "dataFim": data_fim,
                "itens": 100,
                "pagina": pagina,
                "ordem": "ASC",
                "ordenarPor": "dataHoraInicio",
            })
            # sem resposta ou pagina vazia => acabou este deputado/ano
            if not resposta or not resposta["dados"]:
                break
            for d in resposta["dados"]:
                texto = (d.get("transcricao") or "").strip()
                if texto:                  # so guarda discurso COM texto (o que vamos classificar)
                    linhas.append({
                        "id_deputado": dep["id"],
                        "nome_deputado": dep["nome"],
                        "data": d.get("dataHoraInicio"),
                        "tipo": d.get("tipoDiscurso"),
                        "sumario": (d.get("sumario") or "").strip(),
                        "transcricao": texto,
                    })
            pagina += 1
            time.sleep(0.3)   # pausa para nao sobrecarregar o servidor

    # columns=COLS garante cabecalho mesmo quando linhas == [] (deputado sem discurso)
    pd.DataFrame(linhas, columns=COLS).to_csv(caminho, sep=";", index=False, encoding="utf-8")
    print(f"{dep['nome']}: {len(linhas)} discursos com texto")

print("\nColeta concluida. Arquivos em discursos/")

In [ ]:
# Juntar todos os arquivos por deputado em um unico discursos_todos.csv
import glob

partes = [pd.read_csv(f, sep=";", dtype=str) for f in glob.glob("discursos/deputado_*.csv")]
discursos = pd.concat([p for p in partes if len(p)], ignore_index=True)
discursos.to_csv("discursos_todos.csv", sep=";", index=False, encoding="utf-8")

print(f"Total de discursos com texto: {len(discursos)}")
print(f"Deputados com >=1 discurso..: {discursos['id_deputado'].nunique()} de 513")
print("\nTop 10 deputados por nº de discursos (checagem de plausibilidade):")
print(discursos.groupby("nome_deputado").size().sort_values(ascending=False).head(10).to_string())